In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating knowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 76.9 MB/s eta 0:00:00:00:0100:01
Creating knowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [3]:
from transformers import pipeline

zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

result = zs(prompt_150, candidate_labels=labels_150)

for label, score in zip(result['labels'], result['scores']):
    if label == ans_150:
        print(f"Probability assigned to correct answer: {round(score, 3)}")
        break

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Probability assigned to correct answer: 0.384


In [4]:
# Aq2
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])

query_embedding = model.encode([prompt_150])

k = 10
distances, indices = index.search(query_embedding, k)
retrieved_indices = indices[0]
print("Top 10 retrieved KB indices:", retrieved_indices)

if 150 in retrieved_indices:
    rank = list(retrieved_indices).index(150) + 1
    print(f"True document (kb index 150) found at rank: {rank}")
else:
    print("True document not found in top 10")

Top 10 retrieved KB indices: [ 663 1701 1269 1532  576  847 1693 1906  168  150]
True document (kb index 150) found at rank: 10


In [5]:
# Q3

from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs_10 = [kb[i] for i in retrieved_indices]   # top 10 from your Q2 FAISS search
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

# Sort by cross-encoder score, descending
sorted_order = np.argsort(-ce_scores)
sorted_kb_indices = [retrieved_indices[i] for i in sorted_order]

print("Reranked KB indices:", sorted_kb_indices)

if 150 in sorted_kb_indices:
    rank = sorted_kb_indices.index(150) + 1
    print(f"True document reranked to position: {rank}")
else:
    print("True document not in top 10 at all (so reranking can't place it)")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranked KB indices: [np.int64(150), np.int64(1693), np.int64(847), np.int64(1906), np.int64(1269), np.int64(1532), np.int64(168), np.int64(576), np.int64(1701), np.int64(663)]
True document reranked to position: 1


In [6]:
#Q4 
from transformers import AutoTokenizer

row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])

query_embedding_42 = model.encode([prompt_42])
distances_42, indices_42 = index.search(query_embedding_42, k=5)
retrieved_indices_42 = indices_42[0]

docs_5 = [kb[i] for i in retrieved_indices_42]
concatenated_docs = " ".join(docs_5)

rag_string = f"Context: {concatenated_docs} Question: {prompt_42}"

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokens = bert_tokenizer(rag_string, truncation=False)

num_tokens = len(tokens['input_ids'])
print(f"Total tokens: {num_tokens}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Total tokens: 216


Q5

In [7]:
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
true_doc_150 = kb[150]   # the actual correct-answer text stored at KB index 150

labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"

result_rag = zs(rag_string_150, candidate_labels=labels_150)

for label, score in zip(result_rag['labels'], result_rag['scores']):
    if label == ans_150:
        print(f"New probability with RAG context: {round(score, 3)}")
        break

New probability with RAG context: 0.989


In [8]:
# Q6

row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

adversarial_doc = kb[999]   # completely unrelated fact
adversarial_rag_string = f"Context: {adversarial_doc} Question: {prompt_150}"

result_adv = zs(adversarial_rag_string, candidate_labels=labels_150)

for label, score in zip(result_adv['labels'], result_adv['scores']):
    if label == ans_150:
        print(f"Probability with adversarial context: {round(score, 3)}")
        break

Probability with adversarial context: 0.529


In [9]:
# Q7

k = 5
hits = 0

for idx in range(100):
    row = train.iloc[idx]
    prompt = str(row['prompt'])
    correct_letter = row['answer']
    correct_answer_text = str(row[correct_letter])

    query_emb = model.encode([prompt])
    _, retrieved = index.search(query_emb, k)
    retrieved_docs = [kb[i] for i in retrieved[0]]

    # Check if the correct answer text appears in any retrieved doc
    hit = any(correct_answer_text in doc for doc in retrieved_docs)
    if hit:
        hits += 1

hit_rate = round((hits / 100) * 100, 1)
print(f"Hit Rate: {hit_rate}%")

Hit Rate: 73.0%


In [10]:
# q8

def map_at_3_single(true_label, pred_labels):
    if true_label not in pred_labels[:3]:
        return 0
    rank = pred_labels[:3].index(true_label) + 1
    return 1 / rank

scores = []
k = 5

for idx in range(20):
    row = train.iloc[idx]
    prompt = str(row['prompt'])
    true_letter = row['answer']
    candidate_labels = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]

    # Retrieve
    query_emb = model.encode([prompt])
    _, retrieved = index.search(query_emb, k)
    retrieved_idxs = retrieved[0]
    docs_5 = [kb[i] for i in retrieved_idxs]

    # Rerank
    pairs = [[prompt, doc] for doc in docs_5]
    ce_scores = cross_encoder.predict(pairs)
    best_doc = docs_5[np.argmax(ce_scores)]

    # Augment
    rag_string = f"Context: {best_doc} Question: {prompt}"

    # Predict
    result = zs(rag_string, candidate_labels=candidate_labels)

    # Map predicted option texts back to letters, sorted by score descending
    label_to_letter = {str(row[letter]): letter for letter in ["A", "B", "C", "D", "E"]}
    ranked_letters = [label_to_letter[label] for label in result['labels']]

    row_score = map_at_3_single(true_letter, ranked_letters)
    scores.append(row_score)

final_map3 = round(np.mean(scores), 3)
print(f"Final RAG pipeline MAP@3 (20 rows): {final_map3}")

Final RAG pipeline MAP@3 (20 rows): 0.975
